# สำรวจ Response จริงจาก Air4Thai API (`03_inspect_response.ipynb`)

In [1]:
import requests, json
import truststore; truststore.inject_into_ssl()     # แก้ SSLError จาก certificate chain ของ server

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
HEADERS = {"User-Agent": "pm25-explorer/0.1 (student project; baramee.khu.67@ubu.ac.th)"}

r = requests.get(URL, headers=HEADERS, timeout=(10, 60))
print(r.status_code, r.headers.get("Content-Type"), len(r.content), "bytes")
data = r.json()

200 application/json; charset=UTF-8 140124 bytes


In [2]:
print(list(data.keys()))
st = data["stations"]
print(len(st), "สถานี")
print(json.dumps(st[0], ensure_ascii=False, indent=2)[:1500])   # ดูสถานีแรกแบบอ่านง่าย

['stations']
180 สถานี
{
  "stationID": "119t",
  "nameTH": "สวนสาธารณะธารา",
  "nameEN": "Thara Public Park",
  "areaTH": "ต.ปากน้ำ อ.เมือง, กระบี่",
  "areaEN": "Pak Nam Subdistrict, Mueang District, Krabi",
  "stationType": "GROUND",
  "lat": "8.0506237",
  "long": "98.9180489",
  "forecast": [],
  "AQILast": {
    "date": "2026-11-25",
    "time": "10:00",
    "PM25": {
      "color_id": "1",
      "aqi": "12",
      "value": "6.9"
    },
    "PM10": {
      "color_id": "0",
      "aqi": "-1",
      "value": "-1"
    },
    "O3": {
      "color_id": "1",
      "aqi": "6",
      "value": "8"
    },
    "CO": {
      "color_id": "1",
      "aqi": "5",
      "value": "0.80"
    },
    "NO2": {
      "color_id": "1",
      "aqi": "1",
      "value": "3"
    },
    "SO2": {
      "color_id": "1",
      "aqi": "1",
      "value": "2"
    },
    "AQI": {
      "color_id": "1",
      "aqi": "12",
      "param": "PM25"
    }
  }
}


In [3]:
from collections import Counter
Counter(tuple(sorted(s.keys())) for s in st).most_common(3)   # รูปแบบ key ที่พบบ่อย

[(('AQILast', 'areaEN', 'areaTH', 'forecast', 'lat', 'long', 'nameEN', 'nameTH', 'stationID', 'stationType'), 180)]

### ตารางจับคู่ key จาก response จริง

| คอลัมน์ในตารางของเรา | path ของ key ใน JSON | ชนิดใน JSON | ตัวอย่างค่า |
|---|---|---|---|
| `station_id` | `stationID` | `str` | `"119t"` |
| `name_th` / `name_en` | `nameTH` / `nameEN` | `str` | `"สวนสาธารณะธารา"` / `"Thara Public Park"` |
| `province` (อยู่ในข้อความพื้นที่?) | `areaTH` (อยู่ส่วนท้ายหลังเครื่องหมาย `,`) / `areaEN` | `str` | `"ต.ปากน้ำ อ.เมือง, กระบี่"` |
| `lat` / `lon` | `lat` / `long` | `str` | `"8.0506237"` / `"98.9180489"` |
| `station_type` | `stationType` | `str` | `"GROUND"` |
| วันที่ + เวลาของค่าล่าสุด | `AQILast.date` + `AQILast.time` | `str` | `"2026-11-25"` + `"10:00"` |
| `pm25` (ความเข้มข้น ไม่ใช่ AQI) | `AQILast.PM25.value` | `str` | `"6.9"` |
| `pm10`, `o3`, `co`, `no2`, `so2` | `AQILast.<PARAM>.value` (`PM10`, `O3`, `CO`, `NO2`, `SO2`) | `str` | `"-1"`, `"8"`, `"0.80"`, `"3"`, `"2"` |

**คำตอบเพิ่มเติม:**
- **ค่าหายถูกแทนด้วยอะไร:** แทนด้วยสตริง `"-1"` (และอาจพบ `"-999"` หรือ `"-"` ในบางพารามิเตอร์เมื่อเซนเซอร์ไม่ทำงาน)
- **สถานีทุกสถานีมี key ครบเท่ากันหรือไม่:** จาก `Counter` พบว่าทั้ง 180 สถานีมี key ระดับบนสุดครบเท่ากันทั้งหมด 10 key และใน `AQILast` มีครบ 9 key เท่ากันทุกสถานี

In [4]:
from urllib.robotparser import RobotFileParser
rp = RobotFileParser("http://air4thai.pcd.go.th/robots.txt")
rp.read()
rp.can_fetch(HEADERS["User-Agent"], URL)

True